# LAB 6 

### Task 1
Un desarrollador junior de su equipo sugiere: "Para detectar con mayor precisión las texturas de las
hojas enfermas, deberíamos construir una red secuencial clásica (tipo VGG) pero de 150 capas. Más
profundo siempre es mejor". Como líder técnico, explíquele argumentativamente por qué esta red
fracasará estrepitosamente en el entrenamiento (mencionando el fenómeno de degradación y el
desvanecimiento del gradiente). Luego, justifique cómo la adición estructural de las conexiones
residuales (𝐹(𝑥) + 𝑥) de ResNet rescata el proyecto, haciendo viable entrenar redes ultra-profundas
sin colapsar.

# Justificación arquitectónica para evitar una red tipo VGG de 150 capas

Le diría, 

Tu intuición de usar una red más profunda para capturar mejor las texturas de las hojas es normal pensar q entre mas capas, mas sabe.

Pero, construir una **una VGG de 150 capas** introduce problemas de optimización durante el entrenamiento con *backpropagation*.

En particular aparecen dos fenómenos:

- **Vanishing Gradient**
- **Degradation Problem**

Estos problemas hacen que una red de esa profundidad no entrene correctamente.

---

## Vanishing Gradient

Durante el entrenamiento utilizamos *backpropagation* para actualizar los pesos.

El gradiente se propaga hacia atrás multiplicándose a través de muchas capas. Si esos valores son menores que 1, el gradiente se reduce de forma exponencial.

Por ejemplo:

$$
0.9^{150} \approx 1.37 \times 10^{-7}
$$

Esto significa que el gradiente que llega a las primeras capas es **casi cero**.

### Consecuencias

- El aprendizaje en las capas cercanas al input se vuelve mínimo.
- Los pesos de esas capas casi no se actualizan.
- El entrenamiento pierde efectividad.

En una red secuencial de **150 capas**, este efecto aparece con facilidad, especialmente considerando que el dataset de enfermedades de hojas de mango no es grande.

---

## Degradation Problem

Aunque el problema del gradiente se reduzca, aparece otro fenómeno importante: **el problema de degradación**.

Si agregas más capas a una red que ya funciona, el modelo debería poder **copiar la solución anterior**. Esto ocurriría si las capas nuevas aprendieran una función identidad.

Sin embargo, en la práctica ocurre algo como:

| Arquitectura | Error de entrenamiento |
|---------------|-----------------------|
| 20 capas | Bajo |
| 56 capas | Más alto |

Esto indica que el modelo **no logra optimizarse correctamente**.

No se trata de **overfitting**, sino de un problema de optimización. Las redes profundas tienen dificultad para aprender transformaciones cercanas a la identidad.

---

## ResNet

La idea es cambiar lo que aprende cada bloque de la red.

En una red tradicional, las capas intentan aprender una función completa:

$$
H(x)
$$

ResNet propone aprender una **función residual**:

$$
F(x) = H(x) - x
$$

La salida del bloque se calcula como:

$$
y = F(x) + x
$$

Este mecanismo introduce una **skip connection**, donde la entrada del bloque se suma directamente a su salida.

---

## Por qué las conexiones residuales funcionan

Las conexiones residuales ayudan por dos razones principales.

### 1. Mejor flujo del gradiente

El gradiente puede propagarse por dos caminos:

- a través de las capas convolucionales
- a través de la conexión de identidad

Esto reduce el problema de **vanishing gradient** y permite entrenar redes profundas.

---

### 2. Facilitan aprender la identidad

Si un bloque adicional no mejora la representación, la red puede aprender:

$$
F(x) = 0
$$

lo que produce:

$$
y = x
$$

De esta forma, las capas extra **no degradan el rendimiento**, resolviendo el **degradation problem**.

---

## Aplicación al proyecto AgriTech

En el contexto de nuestra aplicación para **detección de enfermedades en hojas de mango**, estas decisiones arquitectónicas son importantes.

Nuestro escenario tiene varias restricciones:

- Dataset limitado  
- Texturas complejas en hojas  
- Recursos de entrenamiento limitados  
- Inferencia en **teléfonos Android de gama baja**  
- Aplicación **100% offline (Edge AI)**  

Una arquitectura basada en **ResNet (por ejemplo ResNet-18 o ResNet-50)** ofrece ventajas:

- Podemos entrenar redes profundas de forma estable.
- Captura patrones visuales en texturas de hojas enfermas.
- Reduce problemas de optimización en redes profundas.
- Produce modelos que luego pueden **optimizarse (quantization o pruning)** para ejecutarse en dispositivos móviles.

Por estas razones, en lugar de construir una **VGG secuencial de 150 capas**, una arquitectura basada en **bloques residuales** es una decisión alineada con las necesidades del proyecto AgriTech.